Load a random repository from the sample and display its feature matrix.

In [ ]:
import sys
import time
from pathlib import Path

project_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "src" / "specmine").is_dir()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "Could not find src/specmine. Open this notebook from the "
        "FinalProject-CS440 folder or one of its subfolders."
    )
sys.path.insert(0, str(project_root / "src"))
print(f"Notebook Python: {sys.executable}")
print(f"Project source: {project_root / 'src'}")

from specmine.data import load_data

In [ ]:
start_time = time.time()
data = load_data(curated_only=True, sample_size=500, seed=42)
if data is None:
    df = None
    print("Dataset load failed; no data is available.")
else:
    df = data
    print(f"\nDataset loaded successfully in {time.time() - start_time:.2f} seconds")
    print(f"Dataset shape: {df.shape}")
    print(f"Dataset columns: {df.columns.tolist()}")
    print(
        f"Unique repositories: {df['repo_name'].nunique() if 'repo_name' in df else 'missing repo_name'}"
    )
    print("Dataset preview:")
    display(df.head(10))

In [ ]:
import traceback

from specmine.data import load_feature_data
from specmine.features import build_features

if df is None:
    print("No data to display because the dataset load failed.")
elif df.empty:
    print("No data to display because the loaded dataset is empty.")
else:
    repo_names = df["repo_name"].dropna().drop_duplicates()

    if repo_names.empty:
        print("No repositories were found in the loaded dataset.")
    else:
        random_repo = repo_names.sample(n=1).iloc[0]
        repo_df = df.loc[df["repo_name"].eq(random_repo)]
        print(f"Random repository: {random_repo}")
        print(f"Repository spec rows: {len(repo_df)}")
        print(f"Repository IDs: {repo_df['file_url_sha16'].tolist()}")
        print("Repository rows preview:")
        display(repo_df.head(10))
        try:
            feature_load_started = time.time()
            feature_rows = load_feature_data(repo_df)
        except Exception as e:
            print(f"\nError loading feature data: {e}")
            print("Check requirements.txt for needed packages")
            traceback.print_exc()
        else:
            feature_matrix = build_features(feature_rows)

            print(
                f"Feature loading and engineering time: {time.time() - feature_load_started:.2f} seconds"
            )
            print(f"Raw feature rows shape: {feature_rows.shape}")
            print(f"Raw feature columns: {feature_rows.columns.tolist()}")
            print(f"Engineered feature columns: {feature_matrix.columns.tolist()}")
            print(f"Feature matrix shape: {feature_matrix.shape}")
            print("Engineered feature matrix preview:")
            display(feature_matrix.head(20))

## Training with labels

Set `SPECMINE_LABELS_CSV` to a CSV with `spec_id,label` before running the training cell. Use `1` if AI was used at any point in creating the spec, `0` if no AI was used, and leave ambiguous cases blank. `examples/synthetic_labels.csv` is fabricated and is only for testing the pipeline; its scores are not authorship accuracy. The notebook has no proxy-label or stub-model fallback.


In [ ]:
import logging
import os

import pandas as pd

from specmine.data import load_data
from specmine.labels import read_labels_csv
from specmine.train import prepare_training_data, train_models

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logging.getLogger("specmine.train").setLevel(logging.INFO)

os.environ["SPECMINE_LABELS_CSV"] = "../examples/synthetic_labels.csv"
labels_path = os.environ.get("SPECMINE_LABELS_CSV")
if not labels_path:
    raise ValueError(
        "Set SPECMINE_LABELS_CSV to a spec_id,label CSV. Use "
        "examples/synthetic_labels.csv only to smoke-test the pipeline; "
        "it is not authorship ground truth. No proxy-label fallback is used."
    )

labels_frame = read_labels_csv(labels_path)
training_data = load_data(seed=42)
if training_data is None:
    raise RuntimeError("SpecMine data loading failed; training cannot continue.")

training_features, training_labels, training_repos = prepare_training_data(
    training_data, labels_frame, seed=42
)
training_results = train_models(
    training_features,
    training_labels,
    training_repos,
    test_size=0.2,
    seed=42,
)
baseline_models = {name: result["model"] for name, result in training_results.items()}
print(
    "Metrics below evaluate only the supplied CSV labels. They are not "
    "authorship accuracy unless those labels are verified ground truth."
)
display(
    pd.DataFrame(
        [
            {
                "model": name,
                "holdout_specs": result["n_test"],
                "holdout_repositories": result["n_test_repositories"],
                **result["metrics"],
            }
            for name, result in training_results.items()
        ]
    ).set_index("model")
)


In [ ]:
import json
import logging
import os
import traceback

import pandas as pd

from specmine.evaluate import (
    _attach_labels,
    _expected_columns,
    _load_model,
    _read_labels,
    evaluate_model,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
debug_logger = logging.getLogger("specmine.notebook.evaluate")
model_path = os.environ.get("SPECMINE_MODEL_PATH")
labels_path = os.environ.get("SPECMINE_LABELS_CSV")

print("=== Evaluation debug setup ===")
print(f"Evaluator module: {__import__('specmine.evaluate', fromlist=['']).__file__}")
print(f"Random repository: {random_repo}")
print(f"Rows in repository metadata: {len(repo_df)}")
print(f"Model path from SPECMINE_MODEL_PATH: {model_path!r}")
print(f"Labels path from SPECMINE_LABELS_CSV: {labels_path!r}")

if model_path:
    model = _load_model(Path(model_path))
    print(f"Loaded trained model: {type(model).__module__}.{type(model).__name__}")
elif "baseline_models" in globals() and "logistic_regression" in baseline_models:
    model = baseline_models["logistic_regression"]
    print("Using the Logistic Regression model trained in the previous cell.")
else:
    raise RuntimeError(
        "No trained model is available. Run the labels-required training cell "
        "or set SPECMINE_MODEL_PATH to a saved model; no stub is used."
    )

if "feature_matrix" not in globals() or feature_matrix.empty:
    raise RuntimeError("The preceding feature cell did not produce a feature_matrix.")
if "file_url_sha16" not in repo_df or "file_url_sha16" not in feature_matrix:
    raise KeyError("Both repository rows and features must contain file_url_sha16.")

evaluation_frame = repo_df.merge(
    feature_matrix,
    on="file_url_sha16",
    how="inner",
    validate="one_to_one",
    suffixes=("", "_feature"),
)
expected_columns = _expected_columns(model)
missing_columns = [
    column for column in expected_columns if column not in evaluation_frame
]
print(f"Merged evaluation frame shape: {evaluation_frame.shape}")
print(f"Merged evaluation columns: {evaluation_frame.columns.tolist()}")
print(f"Missing model feature columns: {missing_columns}")
print("Evaluation frame dtypes:")
print(evaluation_frame.dtypes.to_string())
print("Evaluation frame preview:")
display(evaluation_frame.head(10))
if missing_columns:
    raise ValueError(f"Evaluation frame is missing model columns: {missing_columns}")

if labels_path:
    labels = _read_labels(Path(labels_path))
    print(
        f"Labels loaded: {len(labels)} rows; label counts:\n"
        f"{labels['label'].value_counts(dropna=False)}"
    )
    evaluation_frame = _attach_labels(evaluation_frame, labels)
    n_unmatched = int((~evaluation_frame["_label_matched"]).sum())
    n_unlabeled = int(
        (evaluation_frame["_label_matched"] & evaluation_frame["label"].isna()).sum()
    )
    print(f"Unmatched rows: {n_unmatched}; matched rows missing labels: {n_unlabeled}")
    labeled_frame = evaluation_frame.loc[evaluation_frame["label"].notna()]
    if labeled_frame.empty:
        print("No labeled examples matched this repository; running prediction-only.")
        evaluation_frame = evaluation_frame.drop(columns=["label", "_label_matched"])
    else:
        evaluation_frame = labeled_frame.drop(columns="_label_matched")
else:
    print("No label CSV configured; supervised metrics will be skipped.")

print("=== Running evaluate_model ===")
try:
    evaluation_result = evaluate_model(model, evaluation_frame, threshold=0.5)
except Exception:
    debug_logger.exception("Evaluation failed")
    traceback.print_exc()
    raise

print(json.dumps(evaluation_result, indent=2, default=str))
print("Per-example predictions:")
display(pd.DataFrame(evaluation_result["predictions"]))
if evaluation_result["metrics"] is None:
    print(
        "No matching labels were available for these examples; no metrics computed."
    )
else:
    print(
        "Metrics evaluate the supplied labels only; they are not authorship "
        "accuracy unless those labels are verified ground truth."
    )
    print(json.dumps(evaluation_result["metrics"], indent=2, default=str))
